In [ ]:
import pandas as pd
import numpy as np
import optuna
import mlflow
import mlflow.xgboost
mlflow.set_tracking_uri("file:///C:/Users/phani/sentinel-score-api/mlruns")
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE

# Load advanced featured data
df = pd.read_csv('../data/train_advanced_features.csv')

X = df.drop(columns=['isFraud', 'TransactionID'])
y = df['isFraud']
X = X.fillna(0)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Train shape: {X_train_balanced.shape}")
print(f"Test shape: {X_test.shape}")

c:\Users\phani\anaconda3\envs\sentinelscore\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\phani\anaconda3\envs\sentinelscore\lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


Train shape: (911804, 221)
Test shape: (118108, 221)


In [2]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("sentinelscore-fraud-detection")

def objective(trial):
    # Optuna suggests values to try
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }
    
    # Train model with suggested params
    model = XGBClassifier(**params, eval_metric='auc', n_jobs=-1, random_state=42)
    model.fit(X_train_balanced, y_train_balanced)
    
    # Evaluate
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log to MLflow
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        mlflow.log_params(params)
        mlflow.log_metric("auc_roc", auc)
    
    return auc

print("Objective function defined.")

2026/03/29 20:46:09 INFO mlflow.tracking.fluent: Experiment with name 'sentinelscore-fraud-detection' does not exist. Creating a new experiment.


Objective function defined.


In [3]:
# Create study — maximize AUC
study = optuna.create_study(direction="maximize")

# Run 20 trials — each trial is one experiment with different settings
study.optimize(objective, n_trials=20)

# Best result
print(f"\nBest AUC: {study.best_value:.4f}")
print(f"Best parameters: {study.best_params}")

[I 2026-03-29 20:47:04,633] A new study created in memory with name: no-name-ad3b5d71-cb47-4e26-8ccc-087a3bfca386
[I 2026-03-29 20:48:56,279] Trial 0 finished with value: 0.9013465052946961 and parameters: {'n_estimators': 336, 'max_depth': 7, 'learning_rate': 0.04417822428882138, 'subsample': 0.712870796193144, 'colsample_bytree': 0.9869760868442276, 'min_child_weight': 5}. Best is trial 0 with value: 0.9013465052946961.
[I 2026-03-29 20:50:58,089] Trial 1 finished with value: 0.9562063558218092 and parameters: {'n_estimators': 326, 'max_depth': 10, 'learning_rate': 0.1591172798858977, 'subsample': 0.8088763386471961, 'colsample_bytree': 0.8471602800333115, 'min_child_weight': 4}. Best is trial 1 with value: 0.9562063558218092.
[I 2026-03-29 20:52:56,146] Trial 2 finished with value: 0.9381423290845881 and parameters: {'n_estimators': 323, 'max_depth': 9, 'learning_rate': 0.08395157951496969, 'subsample': 0.6225492570026989, 'colsample_bytree': 0.6704958684000851, 'min_child_weight': 


Best AUC: 0.9610
Best parameters: {'n_estimators': 458, 'max_depth': 10, 'learning_rate': 0.25527828123914464, 'subsample': 0.941099073838524, 'colsample_bytree': 0.8727503141801909, 'min_child_weight': 6}


In [4]:
best_params = study.best_params

# Train final model with best parameters
final_model = XGBClassifier(
    **best_params,
    eval_metric='auc',
    n_jobs=-1,
    random_state=42
)

final_model.fit(X_train_balanced, y_train_balanced)

# Evaluate
y_pred_proba = final_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Final Model AUC: {auc:.4f}")

# Log to MLflow
with mlflow.start_run(run_name="best_model_optuna"):
    mlflow.log_params(best_params)
    mlflow.log_metric("auc_roc", auc)
    mlflow.xgboost.log_model(final_model, "model")

print("Best model logged to MLflow.")

# Save locally
import pickle
with open('../models/xgboost_best.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print("Best model saved to models/")

Final Model AUC: 0.9610


c:\Users\phani\anaconda3\envs\sentinelscore\lib\site-packages\xgboost\sklearn.py:1116: UserWarning: [21:33:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)


Best model logged to MLflow.
Best model saved to models/


In [5]:
mlflow.set_tracking_uri("file:///C:/Users/phani/sentinel-score-api/mlruns")
